In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

API_KEY = "eM9YZzw4eGKAS8iVRtpVyPbhbzYaCW3so-lAWkU61p3rMR-jB_8"

response = requests.get(
    "https://api.pandascore.co/lol/matches/upcoming",
    params={"token": API_KEY, "per_page": 100}
)

matches = response.json()

def calculate_expectedvalue(PW, VW, PL, VL):
    ExV = (PW * VW) - (PL * VL)
    return ExV

def calculate_odds(team1, team2, data, col):
    r1 = max(0.01, min(0.99, data.loc[data['teamname'] == team1, col].iloc[0]))
    r2 = max(0.01, min(0.99, data.loc[data['teamname'] == team2, col].iloc[0]))
    prob = r1 / (r1 + r2)
    return prob
def Exv_Creator()

df = pd.read_csv('2026_LoL_esports_match_data_from_OraclesElixir.csv')
print("Hello, Welcome to the EV calculator")
print("Please choose a region LEC/LCS")
region = input()

if region == "LEC":
    league_2026 = df[(df['league'] == 'LEC') & (df['split'] == 'Spring')]
    print(league_2026.columns.tolist())
    lol_matches = []
    for m in matches:
        if m["league"]["name"] == "LEC":
            lol_matches.append(m)
elif region == "LCS":
    league_2026 = df[(df['league'] == 'LCS') & (df['split'] == 'Spring')]
    print(league_2026.columns.tolist())
    lol_matches = []
    for m in matches:
        if m["league"]["name"] == "LCS":
            lol_matches.append(m)
elif region == "LCK":
    league_2026 = df[(df['league'] == 'LCK') & (df['split'] == 'Spring')]
    print(league_2026.columns.tolist())
    lol_matches = []
    for m in matches:
        if m["league"]["name"] == "LCK":
            lol_matches.append(m)

print("Choose mode : Available options 'WO' for odds to win, FBR 'for first blood chance':")
mode = input()

if mode == 'WO':
    league_2026["team_wo"] = league_2026.groupby("teamid")["result"].transform("mean")
    WOLeaderboard = league_2026[["teamname", "team_wo"]].drop_duplicates(subset="teamname").sort_values(by="team_wo", ascending=False)
    print(WOLeaderboard.head(len(WOLeaderboard)))
    for team in WOLeaderboard["teamname"]:
        print(team)
    print("Enter team1 name:")
    team1_name = input()
    print("Enter team2 name:")
    team2_name = input()
    q = calculate_odds(team1_name, team2_name, WOLeaderboard, "team_wo")
    print("Which team are you betting on?")
    bet_team = input()
    if bet_team == team1_name:
        bet_q = q
    else:
        bet_q = 1 - q
    print(f"Matchup: {team1_name} vs {team2_name}")
    print(f"{team1_name} Win Odds: {q:.1%}")
    print(f"{team2_name} Win Odds: {1 - q:.1%}")

elif mode == 'FBR':
    league_2026['team_fbr'] = league_2026.groupby("teamid")["firstblood"].transform("mean")
    FBRLeaderboard = league_2026[["teamname", "team_fbr"]].drop_duplicates(subset="teamname").sort_values(by="team_fbr", ascending=False)
    print(FBRLeaderboard.head(len(FBRLeaderboard)))
    for team in FBRLeaderboard["teamname"]:
        print(team)
    print("Enter team1 name:")
    team1_name = input()
    print("Enter team2 name:")
    team2_name = input()
    q = calculate_odds(team1_name, team2_name, FBRLeaderboard, "team_fbr")
    print("Which team are you betting on?")
    bet_team = input()
    if bet_team == team1_name:
        bet_q = q
    else:
        bet_q = 1 - q
    print(f"Matchup: {team1_name} vs {team2_name}")
    print(f"{team1_name} Win Odds: {q:.1%}")
    print(f"{team2_name} Win Odds: {1 - q:.1%}")

print("What is the return rate:")
returnrate = float(input())

possible_wagers = range(0, 101, 1)

chart_df = pd.DataFrame({"wager": possible_wagers})
chart_df["EV"] = chart_df["wager"].apply(lambda w: calculate_expectedvalue(bet_q, w * returnrate, 1 - bet_q, w))
chart_df["EV_per10EUR"] = chart_df["EV"] / 10

chart_df.plot(kind='line', x='wager', y='EV_per10EUR',
              figsize=(10, 6), color='blue', linewidth=1)

plt.title('Expected Value per 10 EUR')
plt.xlabel('Wager (EUR)')
plt.ylabel('Expected Value')
plt.grid(True)
plt.show()

print("EV Guide:")
print("  > 0%  : Technically profitable long term")
print("  2-5%  : Good — what sharp bettors aim for")
print("  5-10% : Very good, bookmaker likely mispriced")
print("  10%+  : Exceptional, won't last long")
print("Do you want to know your ev percentage (Yes):")
ans = input()
if ans == "Yes":
    print("How much are u going to wager:")
    wager = float(input())
    ev_percent = (calculate_expectedvalue(q, wager * returnrate, 1 - q, wager) / wager)
    print(f"Your EV percentage is {ev_percent:.1f}%")
